In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────
import os, sys

SPACE = "NeuralHU/forge-rl"   # ← PUT YOUR HF USERNAME HERE

# Clone repo
os.system(f"git clone https://huggingface.co/spaces/{SPACE} /content/forge-rl")
sys.path.insert(0, "/content/forge-rl")   # FIXED: was ./forge-rl
os.chdir("/content/forge-rl")

# Install openenv-core (REQUIRED — was completely missing)
os.system('pip install -q "openenv-core[core]>=0.2.1"')

# Install unsloth correctly for Colab (FIXED: was pip install unsloth which fails)
os.system('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')

# Install remaining deps
os.system("pip install -q trl datasets transformers accelerate peft")
os.system("pip install -q torch-geometric")

print("Setup complete.")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("NO GPU DETECTED. Runtime → Change runtime type → T4 GPU")
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded.")

In [ ]:
# ── Cell 3: Reward function ────────────────────────────────────
# Uses ForgeEnv locally. step() returns (obs, reward, terminated, truncated, info)
# from Gymnasium interface (what ForgeEnv actually uses).
import numpy as np
from env.forge_env import ForgeEnv, ForgeEnvConfig

TOOL_ACTIONS = [0, 5, 2]  # query_source=0, temporal_audit=5, cross_reference=2

def _parse_verdict(text: str) -> int:
    """Map model output text → correct action index per misinfo_env.py."""
    t = text.lower()
    if any(w in t for w in ["fabricat"]):
        return 12   # submit_verdict_fabricated
    if any(w in t for w in ["out of context", "recontextual", "cropped"]):
        return 11   # submit_verdict_out_of_context
    if any(w in t for w in ["satire", "parody", "joke", "humor"]):
        return 10   # submit_verdict_satire
    if any(w in t for w in ["misinfo", "false", "manipulat", "mislead"]):
        return 9    # submit_verdict_misinfo
    if any(w in t for w in ["verified", "true", "legitimate", "accurate", "real"]):
        return 8    # submit_verdict_real
    return 9        # default: misinfo

def _safe_step(env, action):
    """
    FIXED: env.step() always requires an action argument.
    Returns (reward, done). All other fields discarded.
    Handles both Gymnasium 5-tuple and simplified 2-tuple returns defensively.
    """
    result = env.step(action)
    if isinstance(result, tuple):
        if len(result) == 5:
            _, reward, terminated, truncated, _ = result
            return float(reward), bool(terminated or truncated)
        elif len(result) == 4:
            _, reward, terminated, _ = result
            return float(reward), bool(terminated)
        elif len(result) == 2:
            _, reward = result
            return float(reward), False
    # OpenEnv Observation object
    return float(getattr(result, "reward", 0.0)), bool(getattr(result, "done", False))

def _safe_reset(env):
    """
    FIXED: env.reset() return type varies.
    Returns (obs, info) regardless of underlying implementation.
    """
    result = env.reset()
    if isinstance(result, tuple) and len(result) == 2:
        return result  # Gymnasium (obs, info)
    return result, {}  # OpenEnv single Observation

def reward_fn(prompts, completions, claim_texts=None, **kwargs):
    """
    GRPO reward function — FIXED VERSION.
    Each prompt gets a fresh ForgeEnv episode.
    Claim is overridden if claim_texts kwarg is provided.
    Returns List[float] clipped to (0.001, 0.999).
    """
    rewards = []
    for i, completion in enumerate(completions):
        comp_text = (
            completion[-1]["content"]       # chat format: last message is model output
            if isinstance(completion, list)
            else str(completion)
        )
        try:
            cfg = ForgeEnvConfig(budget=6, seed=42 + i)
            env = ForgeEnv(cfg)
            obs, info = _safe_reset(env)

            # Override claim if dataset provided one
            # FIXED: use only public-facing attributes, no private method calls
            if claim_texts and i < len(claim_texts):
                claim = claim_texts[i]
                # Set claim text on env if attribute exists, else skip
                if hasattr(env, "_claim_text"):
                    env._claim_text = claim
                if hasattr(env, "_claim_text_initial"):
                    env._claim_text_initial = claim

            # Run investigation tools
            # FIXED: env.step() ALWAYS requires an action argument
            done = False
            for act in TOOL_ACTIONS:
                if done:
                    break
                _, done = _safe_step(env, act)

            # Submit verdict from model completion
            verdict_action = _parse_verdict(comp_text)
            reward, _ = _safe_step(env, verdict_action)
            rewards.append(float(np.clip(reward, 0.001, 0.999)))

        except Exception as e:
            print(f"[reward_fn] episode {i} error: {e}")
            rewards.append(0.001)

    return rewards

print("reward_fn ready.")

In [ ]:
# ── Cell 4: Dataset ────────────────────────────────────────────
from datasets import Dataset
import random

INVESTIGATION_PROMPT_TEMPLATE = """\
You are a misinformation forensics agent. You have investigated the following claim:

CLAIM: {claim}

Your investigation found:
- Source credibility signals from query_source
- Temporal consistency from temporal_audit
- Cross-reference matches from cross_reference

Based on this investigation, submit your verdict. Respond with exactly one of:
- "This is MISINFO because [reason]"
- "This is SATIRE because [reason]"
- "This is VERIFIED because [reason]"
- "This is FABRICATED because [reason]"
- "This is OUT OF CONTEXT because [reason]"

Your verdict:"""

TASK_NAMES = [
    "fabricated_stats", "out_of_context", "coordinated_campaign",
    "satire_news", "verified_fact", "politifact_liar",
    "image_forensics", "sec_fraud",
]

# FIXED: safe claim getter — no longer calls MisInfoForensicsEnv with wrong args
def _get_claim_for_task(task_name: str, seed: int) -> str:
    """Get a claim from the task generator. Falls back gracefully."""
    try:
        from env.forge_env import ForgeEnv, ForgeEnvConfig
        cfg = ForgeEnvConfig(budget=3, seed=seed)
        env = ForgeEnv(cfg)
        obs, info = _safe_reset(env)   # uses the _safe_reset from Cell 3
        # Try common attribute names for claim text
        for attr in ("_claim_text", "claim_text"):
            val = getattr(env, attr, None)
            if val:
                return str(val)
        # Try from info dict
        if info and "claim_text" in info:
            return info["claim_text"]
        return f"Unverified claim #{seed} in domain: {task_name}"
    except Exception:
        return f"Unverified claim #{seed} in domain: {task_name}"

N = 200
random.seed(42)
claims, prompts = [], []
for i in range(N):
    task = TASK_NAMES[i % len(TASK_NAMES)]
    claim = _get_claim_for_task(task, seed=i)
    claims.append(claim)
    prompts.append([{
        "role": "user",
        "content": INVESTIGATION_PROMPT_TEMPLATE.format(claim=claim)
    }])

# claim_texts column passed as kwarg to reward_fn automatically by TRL GRPO
dataset = Dataset.from_dict({"prompt": prompts, "claim_texts": claims})
print(f"Dataset: {len(dataset)} rows across {len(TASK_NAMES)} task types")
print(f"Sample claim: {claims[0][:100]}...")

In [ ]:
# ── Cell 5: Baseline evaluation BEFORE training ────────────────
import numpy as np

def evaluate_heuristic(n_episodes=20, default_verdict=9):
    """
    Heuristic baseline: always submit one fixed verdict.
    FIXED: passes action to every env.step() call.
    """
    rewards, correct = [], 0
    from env.forge_env import ForgeEnv, ForgeEnvConfig
    for i in range(n_episodes):
        try:
            cfg = ForgeEnvConfig(budget=6, seed=100 + i)
            env = ForgeEnv(cfg)
            _safe_reset(env)
            done = False
            for act in TOOL_ACTIONS:
                if done:
                    break
                _, done = _safe_step(env, act)       # FIXED: action argument required
            reward, _ = _safe_step(env, default_verdict)
            rewards.append(float(reward))
            if reward > 0.5:
                correct += 1
        except Exception as e:
            print(f"Eval error ep {i}: {e}")
            rewards.append(0.001)
    return float(np.mean(rewards)), correct / n_episodes

baseline_reward, baseline_acc = evaluate_heuristic()
print(f"HEURISTIC BASELINE — reward: {baseline_reward:.4f} | acc: {baseline_acc:.2%}")
print("(Always-misinfo heuristic. Trained agent should beat this.)")

In [ ]:
# ── Cell 6: GRPO Training ──────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="./forge-grpo",
    max_steps=150,
    num_generations=4,
    max_completion_length=128,
    per_device_train_batch_size=1,       # FIXED: was 1 — keep 1 for T4 16GB
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    save_steps=50,
    logging_steps=5,
    report_to="none",                    # FIXED: string not empty list
    warmup_ratio=0.1,
)

# FIXED: use processing_class for TRL>=0.9, fall back to tokenizer for older
try:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_fn,
        args=config,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_fn,
        args=config,
        train_dataset=dataset,
        tokenizer=tokenizer,
    )

print("Starting GRPO training (max 150 steps ~45 min on T4)...")
trainer.train()
print("Training complete.")

In [ ]:
# ── Cell 7: Post-training evaluation + plots ───────────────────
import matplotlib.pyplot as plt, matplotlib, os, numpy as np
matplotlib.use('Agg')
os.makedirs("results", exist_ok=True)

def evaluate_trained(n_episodes=20):
    """Run the trained model on new claims and score with ForgeEnv."""
    rewards, correct = [], 0
    from env.forge_env import ForgeEnv, ForgeEnvConfig
    FastLanguageModel.for_inference(model)

    for i in range(n_episodes):
        try:
            task = TASK_NAMES[i % len(TASK_NAMES)]
            claim = _get_claim_for_task(task, seed=200 + i)
            prompt_text = INVESTIGATION_PROMPT_TEMPLATE.format(claim=claim)
            prompt_ids = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt_text}],
                tokenize=True,
                return_tensors="pt",
                add_generation_prompt=True,
            ).to("cuda")
            outputs = model.generate(
                input_ids=prompt_ids,
                max_new_tokens=80,
                temperature=0.3,
                do_sample=True,
            )
            prompt_len = prompt_ids.shape[1]
            response = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
            verdict_action = _parse_verdict(response)

            cfg = ForgeEnvConfig(budget=6, seed=200 + i)
            env = ForgeEnv(cfg)
            _safe_reset(env)
            done = False
            for act in TOOL_ACTIONS:
                if done: break
                _, done = _safe_step(env, act)      # FIXED: action arg always required
            reward, _ = _safe_step(env, verdict_action)
            rewards.append(float(reward))
            if reward > 0.5:
                correct += 1
        except Exception as e:
            print(f"Trained eval error ep {i}: {e}")
            rewards.append(0.001)
    return float(np.mean(rewards)), correct / n_episodes

trained_reward, trained_acc = evaluate_trained()
print(f"BASELINE  — reward: {baseline_reward:.4f} | acc: {baseline_acc:.2%}")
print(f"TRAINED   — reward: {trained_reward:.4f} | acc: {trained_acc:.2%}")
print(f"DELTA     — {trained_reward-baseline_reward:+.4f} | {trained_acc-baseline_acc:+.2%}")

# FIXED: correct TRL GRPO log key is "rewards/mean" (with 's')
log_history = trainer.state.log_history
# Try both key variants defensively
steps, rew = [], []
for l in log_history:
    for key in ("rewards/mean", "reward/mean", "reward"):
        if key in l:
            steps.append(l["step"])
            rew.append(l[key])
            break

fig, ax = plt.subplots(figsize=(10, 5))
if steps:
    ax.plot(steps, rew, 'b-', linewidth=2.5, label='Training reward')
ax.axhline(y=baseline_reward, color='r', linestyle='--',
           linewidth=2, label=f'Heuristic baseline ({baseline_reward:.3f})')
ax.set_xlabel("Training Step", fontsize=13)
ax.set_ylabel("Mean Episode Reward", fontsize=13)
ax.set_title("FORGE-MA: LLM Agent Learning Misinformation Detection via GRPO",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("results/reward_curve.png", dpi=150, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['Heuristic Baseline', 'GRPO Trained'],
              [baseline_reward, trained_reward],
              color=['#e74c3c', '#2ecc71'], edgecolor='black', width=0.5)
ax.set_ylabel("Mean Episode Reward", fontsize=13)
ax.set_title("Before vs After GRPO\n(FORGE-MA Blue Team Agent)", fontsize=13)
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, [baseline_reward, trained_reward]):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("results/before_after.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: results/reward_curve.png + results/before_after.png")
print("COMMIT BOTH TO REPO BEFORE SUBMITTING")